### RAG Pipelines-Data Ingestion to Vector DB Pipeline


In [4]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\KK SYSTEMS\AppData\Local\Temp\ipykernel_6112\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [5]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
                
            all_documents.extend(documents)
        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")
            
    return all_documents

#process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing: 2022239006_caleb_durai (2).pdf

Processing: Apriori AlgorithmExample.pdf

Processing: BDA_Assignment II_Sheet.pdf

Processing: BDA_Assign_1TO8_PROBLEMS.pdf


In [6]:
all_pdf_documents

[Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-02-16T19:06:06+00:00', 'title': '2022239006_caleb_durai', 'moddate': '2026-02-16T19:06:05+00:00', 'keywords': 'DAGWEyWc2D0,BAEbiHsucXw,0', 'author': 'Caleb durai b anna', 'source': '..\\data\\pdf\\2022239006_caleb_durai (2).pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': '2022239006_caleb_durai (2).pdf', 'file_type': 'pdf'}, page_content='SMART HOME MANAGEMENTSYSTEM USING IOT\nName        : Caleb durai B Anna\nReg. No    : 2022239006\nBranch      : Computer Science\nSubject     : Creative And Innovative Project\nSub Code : XC 5811\nDate            : 17 Feb 2026'),
 Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-02-16T19:06:06+00:00', 'title': '2022239006_caleb_durai', 'moddate': '2026-02-16T19:06:05+00:00', 'keywords': 'DAGWEyWc2D0,BAEbiHsucXw,0', 'author': 'Caleb durai b anna', 'source': '..\\data\\pdf\\2022239006_caleb_durai (2).pdf', 'total_pages

In [7]:
### Text splitting get into chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents, chunk_size=100, chunk_overlap=20):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    # Split the loaded LangChain documents
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
        
    return split_docs

In [8]:
chunks=split_documents(all_pdf_documents)

Split 43 documents into 174 chunks

Example chunk:
Content: SMART HOME MANAGEMENTSYSTEM USING IOT
Name        : Caleb durai B Anna
Reg. No    : 2022239006...
Metadata: {'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-02-16T19:06:06+00:00', 'title': '2022239006_caleb_durai', 'moddate': '2026-02-16T19:06:05+00:00', 'keywords': 'DAGWEyWc2D0,BAEbiHsucXw,0', 'author': 'Caleb durai b anna', 'source': '..\\data\\pdf\\2022239006_caleb_durai (2).pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': '2022239006_caleb_durai (2).pdf', 'file_type': 'pdf'}


### Embedding and vectorStoreDB


In [9]:
import numpy as np
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Any, Tuple, Dict
from sklearn.metrics.pairwise import cosine_similarity

In [10]:

from sentence_transformers import SentenceTransformer

class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully.")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
            
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

## initiate the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3143.86it/s]


Model loaded successfully.


VectorStore

In [11]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 174


In [12]:
chunks

[Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-02-16T19:06:06+00:00', 'title': '2022239006_caleb_durai', 'moddate': '2026-02-16T19:06:05+00:00', 'keywords': 'DAGWEyWc2D0,BAEbiHsucXw,0', 'author': 'Caleb durai b anna', 'source': '..\\data\\pdf\\2022239006_caleb_durai (2).pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': '2022239006_caleb_durai (2).pdf', 'file_type': 'pdf'}, page_content='SMART HOME MANAGEMENTSYSTEM USING IOT\nName        : Caleb durai B Anna\nReg. No    : 2022239006'),
 Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-02-16T19:06:06+00:00', 'title': '2022239006_caleb_durai', 'moddate': '2026-02-16T19:06:05+00:00', 'keywords': 'DAGWEyWc2D0,BAEbiHsucXw,0', 'author': 'Caleb durai b anna', 'source': '..\\data\\pdf\\2022239006_caleb_durai (2).pdf', 'total_pages': 9, 'page': 0, 'page_label': '1', 'source_file': '2022239006_caleb_durai (2).pdf', 'file_type': 'pdf'}, page_content='Branch    

In [13]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 174 texts...


Batches: 100%|██████████| 6/6 [00:02<00:00,  2.48it/s]


Generated embeddings with shape: (174, 384)
Adding 174 documents to vector store...
Successfully added 174 documents to vector store
Total documents in collection: 348


Retriever Pipeline From VectorStore

In [14]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [15]:
rag_retriever


In [16]:
rag_retriever.retrieve("What is Apriori")

Retrieving documents for query: 'What is Apriori'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 10.76it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_03d65942_51',
  'content': 'Apriori Algorithm:',
  'metadata': {'doc_index': 51,
   'page_label': '1',
   'source': '..\\data\\pdf\\Apriori AlgorithmExample.pdf',
   'file_type': 'pdf',
   'page': 0,
   'content_length': 18,
   'creator': 'Microsoft® Word 2016',
   'creationdate': '2025-02-13T16:22:50+00:00',
   'total_pages': 3,
   'source_file': 'Apriori AlgorithmExample.pdf',
   'producer': 'www.ilovepdf.com',
   'author': 'Sundar S',
   'moddate': '2025-02-13T16:22:50+00:00'},
  'similarity_score': 0.4910895824432373,
  'distance': 0.5089104175567627,
  'rank': 1},
 {'id': 'doc_d586ee3c_51',
  'content': 'Apriori Algorithm:',
  'metadata': {'source_file': 'Apriori AlgorithmExample.pdf',
   'creator': 'Microsoft® Word 2016',
   'total_pages': 3,
   'producer': 'www.ilovepdf.com',
   'file_type': 'pdf',
   'page': 0,
   'content_length': 18,
   'page_label': '1',
   'source': '..\\data\\pdf\\Apriori AlgorithmExample.pdf',
   'doc_index': 51,
   'creationdate': '2025-02-1

Integration Vectordb Context pipeline With LLM output

In [19]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-70b-versatile",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [20]:
answer=rag_simple("Solve any problem regarding Apriori Algorithm",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'Solve any problem regarding Apriori Algorithm'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 26.50it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


To solve a problem regarding the Apriori Algorithm, we need to follow these steps:

1. **Define the problem**: Identify the transactional dataset and the minimum support threshold.
2. **Generate candidate itemsets**: Use the Apriori property to generate candidate itemsets of size 1, 2, 3, and so on.
3. **Calculate support counts**: Calculate the support count for each candidate itemset.
4. **Prune itemsets**: Prune itemsets with support counts less than the minimum support threshold.
5. **Generate association rules**: Generate association rules from the frequent itemsets.
6. **Evaluate rules**: Evaluate the generated rules based on confidence and lift.

Example: Suppose we have a transactional dataset of customer purchases, and we want to find association rules with a minimum support of 0.1 and minimum confidence of 0.5. We can use the Apriori Algorithm to generate rules such as "Bread → Milk" with a support of 0.12 and confidence of 0.6.


Enhanced RAG Pipeline Features

In [23]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output
# Example usage:
result = rag_advanced("Steps to solve Apriori Algorithm", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Steps to solve Apriori Algorithm'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 21.25it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: The steps to solve the Apriori Algorithm are not provided in the given context, as the section "Steps of the Apriori Algorithm" is empty. However, the general steps of the Apriori Algorithm are:

1. Data Preparation
2. Find Frequent Itemsets
3. Generate Association Rules
4. Prune Association Rules
5. Evaluate Association Rules

Please note that the exact steps may vary depending on the specific implementation or context.
Sources: [{'source': 'Apriori AlgorithmExample.pdf', 'page': 0, 'score': 0.8250882774591446, 'preview': 'Apriori Algorithm:...'}, {'source': 'Apriori AlgorithmExample.pdf', 'page': 0, 'score': 0.8250882774591446, 'preview': 'Apriori Algorithm:...'}, {'source': 'Apriori AlgorithmExample.pdf', 'page': 0, 'score': 0.7679643779993057, 'preview': 'Key Parameters: \n \nSteps of the Apriori Algorithm:...'}]
Confidence: 0.8250882774591446
Context Preview: Apriori Algorithm:

Apriori Algorithm:

Key Parameters: 
 
Steps of the Apriori Algorithm:


In [25]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("Apriori Algorithm", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'Apriori Algorithm'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 24.86it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Apriori Algorithm:

Apriori Algorithm:

Key Parameters: 
 
Steps of the Apriori Algorithm:

Question: Apriori Algorithm

Answer:



Final Answer: The Apriori Algorithm is a popular association rule learning algorithm used for finding frequent itemsets and generating association rules in a dataset.

Citations:
[1] Apriori AlgorithmExample.pdf (page 0)
[2] Apriori AlgorithmExample.pdf (page 0)
[3] Apriori AlgorithmExample.pdf (page 0)
Summary: The Apriori Algorithm is a widely used algorithm for association rule learning, which helps in discovering patterns and relationships within a dataset. It is specifically designed to find frequent itemsets and generate association rules, making it a valuable tool for data analysis and mining applications.
History: {'question': 'Apriori Algorithm', 'answer': 'The Apriori Algorithm is a popular association rule learning algorithm used for finding frequent itemsets and generating association rules in a dataset.', 'sources': [{'source': 'Apriori AlgorithmExample.pdf', 'page': 0, 'score': 0.9873167322948575, 'preview': 'Apriori Algorithm:...'}, {'source': 'Apriori AlgorithmExample.